# InspectorAI v2 — Treinamento no Google Colab

Este notebook usa o pipeline revisado: dados JSONL curados, estados de revisão explícitos, divisão por `group_id` e um único Transformer Encoder–Decoder.

**Não use o conjunto inicial de 18 exemplos para treinar um modelo de uso real.** Ele serve para testar o pipeline. Antes do treinamento, amplie e revise o conjunto (meta inicial: pelo menos 400 exemplos no split de treino, com cobertura adequada de vague, partial e clear).

Envie os arquivos: `train.py`, `prepare_dataset.py`, `model.py`, `tokenizer.py` e `curated_examples.jsonl`. Os dois scripts modificados devem ser os desta versão.


In [ ]:
# 1. Verifique se o runtime está com GPU
import torch
print("PyTorch:", torch.__version__)
print("CUDA disponível:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("Ative Runtime > Change runtime type > GPU e reconecte.")
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))


In [ ]:
# 2. Instale a dependência do tokenizador
%pip install -q tokenizers==0.23.3


In [ ]:
# 3. Monte o Google Drive e envie os cinco arquivos
from google.colab import drive, files
drive.mount("/content/drive")

print("Selecione train.py, prepare_dataset.py, model.py, tokenizer.py e curated_examples.jsonl")
uploaded = files.upload()
required = {"train.py", "prepare_dataset.py", "model.py", "tokenizer.py", "curated_examples.jsonl"}
missing = required - set(uploaded)
if missing:
    raise FileNotFoundError(f"Arquivos não enviados: {sorted(missing)}")
print("Arquivos recebidos:", sorted(required))


In [ ]:
# 4. Organize o projeto e configure a pasta persistente de modelos
from pathlib import Path
import shutil

ROOT = Path("/content/InspectorAI")
SCRIPTS = ROOT / "scripts"
RAW = ROOT / "inspector" / "data" / "raw"
PROCESSED = ROOT / "inspector" / "data" / "processed"
MODELS = ROOT / "inspector" / "models"
for folder in (SCRIPTS, RAW, PROCESSED):
    folder.mkdir(parents=True, exist_ok=True)

for name in ("train.py", "prepare_dataset.py", "model.py", "tokenizer.py"):
    shutil.copy2(Path("/content") / name, SCRIPTS / name)
shutil.copy2(Path("/content/curated_examples.jsonl"), RAW / "curated_examples.jsonl")

DRIVE_MODELS = Path("/content/drive/MyDrive/InspectorAI/v2/models")
DRIVE_MODELS.mkdir(parents=True, exist_ok=True)
MODELS.parent.mkdir(parents=True, exist_ok=True)
if MODELS.is_symlink():
    MODELS.unlink()
elif MODELS.exists():
    shutil.copytree(MODELS, DRIVE_MODELS, dirs_exist_ok=True)
    shutil.rmtree(MODELS)
MODELS.symlink_to(DRIVE_MODELS, target_is_directory=True)

print("Projeto:", ROOT)
print("Dados brutos:", RAW)
print("Modelos persistentes:", DRIVE_MODELS)


In [ ]:
# 5. Valide os registros e gere train/validation/test
%cd /content/InspectorAI
!python scripts/prepare_dataset.py


In [ ]:
# 6. Inspecione a divisão ANTES do treinamento
import json
report_path = PROCESSED / "dataset_report.json"
report = json.loads(report_path.read_text(encoding="utf-8"))
print(json.dumps(report, ensure_ascii=False, indent=2))

train_count = report["counts"]["train"]
if train_count < 400:
    raise RuntimeError(
        f"Há apenas {train_count} exemplos de treino. "
        "Não treine ainda: revise e amplie o dataset antes de continuar."
    )


In [ ]:
# 7. Treine o modelo usando o split preparado
!python scripts/train.py


In [ ]:
# 8. Confirme os artefatos persistidos no Drive
for path in sorted(DRIVE_MODELS.iterdir()):
    if path.is_file():
        print(f"{path.name}: {path.stat().st_size / (1024 * 1024):.2f} MB")


In [ ]:
# 9. Baixe o checkpoint e o tokenizador para substituir o par local
from google.colab import files
model_path = DRIVE_MODELS / "inspector_model.pth"
tokenizer_path = DRIVE_MODELS / "tokenizer_vocab.json"
metrics_path = DRIVE_MODELS / "training_metrics.json"
for path in (model_path, tokenizer_path, metrics_path):
    if not path.exists():
        raise FileNotFoundError(path)
    files.download(str(path))
